In [0]:
data = [(1, "Manasa", 23), (2, "Ravi", 28), (3, "Anu", 25)]
schema = ["id", "name" , "age"]

df = spark.createDataFrame(data  , schema)
df.show()

In [0]:
from pyspark.sql.types import *

data = [(1, "Manasa", 23), (2, "Ravi", 28), (3, "Anu", 25)]
schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("age", IntegerType())
])

df = spark.createDataFrame(data, schema)
df.show()

In [0]:
# Create DF from List of Dictionaries

data = [
 {"id": 1, "name": "A", "salary": "1k"},
 {"id": 2, "name": "B", "salary": "$2000"}
]

schema = StructType([
    StructField("id", IntegerType()),
    StructField("name", StringType()),
    StructField("salary", StringType())
])

df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

In [0]:
# so salary ins tr , it shoul be inthe int
from pyspark.sql.functions import *

# salary_int = df.withColumn("salary" , col("salary").cast("int"))

# edge cases like 2k or $20000

clean_df = df.withColumn(
    "salary_clean",
    when(col("salary").contains("k"),
        regexp_replace(col("salary"), "k" , "").cast("int")*1000 )
    .otherwise(
        regexp_replace(col("salary"), "[^0-9]" , "").cast("int")
    )
)

clean_df.show()
salary_int.printSchema()




In [0]:
# read file from csv :

# df = spark.read("csv").option("header" , True).path("/Volumes/workspace/default/my_files/products-1000.csv")

# / infer shcme i socslty beacuse it reads each ros : use always use schemea 

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("salary", IntegerType(), True)
])

df = spark.read.csv("/Volumes/workspace/default/my_files/products-1000.csv", header=True, schema=schema)
df.printSchema()
display(df)


In [0]:
# 1. Convert STRING → DATE

from pyspark.sql.functions import *

df = spark.createDataFrame([("2025-03-18",)], ["date_str"])

todate_df = df.withColumn("date" , to_date(col("date_str"), "yyyy-MM-dd")) \
            .withColumn("year" , year(col("date"))) \
            .withColumn("month" , month(col("date"))) \
            .withColumn("day" , dayofmonth(col("date")))


todate_df.show()
todate_df.printSchema()
#))

In [0]:
# date_str = "18-03-2025"

# yyyy-MM-dd → ❌ NULL

# dd-MM-yyyy → ✅ valid

df_c = df.withColumn(
    "date",
    coalesce(
        to_date(col("date_str"), "yyyy-MM-dd"),
        to_date(col("date_str"), "dd-MM-yyyy")
    )
)
df_c.show()
df_c.printSchema()


In [0]:
# Q5: Multiple Type Casting

data = [
 (1, "100"),
 (2, "200.5")
]

dff = spark.createDataFrame(data, ["id", "amount"])
dff.show()
dff.printSchema()

In [0]:
# from pyspark.sql.functions import expr , try_cast

# dff_new = dff.withColumn("amt_int",
#     try_cast(col("amount"), "int")) \
#     .withColumn("amt_double" , col("amount").cast("double"))
# dff_new.printSchema()
# dff_new.show()

from pyspark.sql.functions import expr

df = df.withColumn("amount_int", expr("try_cast(amount as int)"))
df.show()

In [0]:
rdd = spark.sparkContext.parallelize([1,2,3,4,5])

total = rdd.reduce(lambda x, y: x + y)
print(total)